In [1]:
# Librerias
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import numpy as np

In [2]:
# =================
# Cargar dataset
# =================

ruta_csv = os.path.join("..", "data", "processed","ESdE_2023_renombrado.csv" )
df_adulto = pd.read_csv(ruta_csv, sep=",", encoding="latin-1")

print("Dataset cargado")

Dataset cargado


In [3]:
targets = ['Hipertension_Diag_Medico', 'Diabetes_Diag_Medico']

for col in targets:
    # A. Eliminar únicamente los '9' (NS/NC) protegiendo los NaN
    df_adulto = df_adulto[
        df_adulto[col].isin([1.0, 2.0, 1, 2]) | df_adulto[col].isna()
    ].copy()

    # B. Convertir los 1 en 1, y todo lo demás (2.0 y NaN) en 0
    df_adulto[col] = (df_adulto[col] == 1).astype(int)

# Comprobación de las clases binarias finales
print("=== HIPERTENSIÓN (0 = Sano, 1 = Enfermo) ===")
print(df_adulto['Hipertension_Diag_Medico'].value_counts())

print("\n=== DIABETES (0 = Sano, 1 = Enfermo) ===")
print(df_adulto['Diabetes_Diag_Medico'].value_counts())

=== HIPERTENSIÓN (0 = Sano, 1 = Enfermo) ===
Hipertension_Diag_Medico
0    15226
1     5772
Name: count, dtype: int64

=== DIABETES (0 = Sano, 1 = Enfermo) ===
Diabetes_Diag_Medico
0    18996
1     2002
Name: count, dtype: int64


In [4]:
# ===========================================
# PREPARACIÓN PARA EL MODELO DE HIPERTENSIÓN
# ===========================================

# Target: Hipertensión
y_hta =df_adulto["Hipertension_Diag_Medico"]

# Separamos la variable objetivo
X_hta = df_adulto.drop(columns=["Hipertension_Diag_Medico"])

#Split estratificado 
X_train_hta, X_test_hta, y_train_hta, y_test_hta = train_test_split(
    X_hta,
    y_hta,
    test_size=0.20,
    random_state=42,
    stratify=y_hta
)

# =========================================
# PREPARACIÓN PARA EL MODELO DE DIABETES
# ========================================

# Target: Diabetes
y_diab = df_adulto["Diabetes_Diag_Medico"]

# Separamos la variable objetivo
X_diab =df_adulto.drop(columns=["Diabetes_Diag_Medico"])

# Split estratificado
X_train_diab, X_test_diab, y_train_diab, y_test_diab = train_test_split(
    X_diab,
    y_diab,
    test_size=0.20,
    random_state=42,
    stratify=y_diab
)

print("Divisiones Train y Test creadas con éxito para ambas enfermedades")

Divisiones Train y Test creadas con éxito para ambas enfermedades


# CONSOLIDAR VARIABLES DE TABACO

FUMADOR DIARIO ACTUAL

Si -> "Tabaquismo" == 1; and "Duracion_Cons_Tabaco" < 5 años 
    entonces "Fumador_Corto_Plazo"

Si -> "Tabaquismo" == 1; and "Duracion_Cons_Tabaco" > 5 años and < 15 años
    entonces "Fumador_Mediano_Plazo"

Si -> "Tabaquismo" == 1; and "Duracion_Cons_Tabaco" > 15 años
    entonces "Fumador_Largo_Plazo"

FUMADOR OCASIONAL ACTUAL

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" < 5 años 
    entonces "Fumador_Ocasional_Bajo_Historial"

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco"> 5 años and < 15 años
    entonces "Fumador_Ocasional_Mediano_Historial"

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" > 15 años
    entonces "Fumador_Ocasional_Alto_Historial"

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 2; 
    entonces "Fumador_Ocasional"

Si -> "Tabaquismo" == 2; and "Ex_Fumador" == 9;
    entonces "Fumador_Ocasional"

EX-FUMADOR

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" < 5 años 
    entonces "Fumador_Corto_Plazo"

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" > 5 años and < 15 años
    entonces "Fumador_Mediano_Plazo"

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 1; and "Duracion_Cons_Tabaco" > 15 años
    entonces "Fumador_Largo_Plazo"

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 2; 
    entonces "Fumador_Ocasional"

Si -> "Tabaquismo" == 3; and "Ex_Fumador" == 9; 
    entonces "Fumador_Ocasional"

NUNCA FUMADOR

Si -> "Tabaquismo" == 4;
    entonces "No_Fumador"

NO CONTESTA

Si -> "Tabaquismo" == 9;
    entonces "No_Contesta"

In [ ]:
# ===================================================
# FUNCIÓN QUE CONSOLIDA LAS TRES VARIABLES DE TABACO
# ===================================================

def consolidar_tabaco(row):
    tab = row.get("Tabaquismo")
    ex = row.get("Ex_Fumador")
    dur = row.get("Duracion_Cons_Tabaco")
    
    # No sabe / No contesta en la pregunta principal
    if tab == 9 or pd.isna(tab):
        return "No_Contesta"
    
    # Nunca fumador
    if tab == 4:
        return "No_Fumador"
    
    # Gestionar nulos y códigos especiales (99) en la duración
    es_dur_desconocida = pd.isna(dur) or dur in [99, 99.0]
    dur_num = 0 if es_dur_desconocida else float(dur)
    
    # Fumador diario actual
    if tab == 1:
        if es_dur_desconocida:
            return "Fumador_Diario_Corto_Plazo"
        if dur_num < 5:
            return "Fumador_Diario_Corto_Plazo"
        elif 5 <= dur_num <= 15:
            return "Fumador_Diario_Mediano_Plazo"
        else:
            return "Fumador_Diario_Largo_Plazo"
        
    # Fumador ocasional actual
    if tab == 2:
        if not es_dur_desconocida and dur_num > 15:
            return "Fumador_Ocasional_Alto_Historial"
        elif not es_dur_desconocida and 5 <= dur_num <= 15:
            return "Fumador_Ocasional_Medio_Historial"
        else:
            return "Fumador_Ocasional"
        
    # Ex-Fumador
    if tab == 3:
        if ex ==1:
            if es_dur_desconocida:
                return "Ex_Fumador_Corto_Plazo"
            if dur_num < 5:
                return "Ex_Fumador_Corto_Plazo"
            elif 5 <= dur_num <= 15:
                return "Ex_Fumador_Mediano_Plazo"
            else: 
                return "Ex_Fumador_Largo_Plazo"
        else:
            return "Ex_Fumador_Ocasional"
    
    return "No_Fumador"
    

In [ ]:
# ==========================================
# AGREGAMOS LA FUNCIÓN DE TABACO A LOS DATOS
# ==========================================


# Aplicar función al conjunto de Hipertensión
X_train_hta["Tabaco_Consolidado"] = X_train_hta.apply(consolidar_tabaco, axis=1)
X_test_hta["Tabaco_Consolidado"] = X_test_hta.apply(consolidar_tabaco, axis=1)

# Aplicar función al conjunto de Diabetes
X_train_diab["Tabaco_Consolidado"] = X_train_diab.apply(consolidar_tabaco, axis=1)
X_test_diab["Tabaco_Consolidado"] = X_test_diab.apply(consolidar_tabaco, axis=1)

# SELECCIÓN DE VARIABLES PARA EL MODELO

In [7]:
# =================================
# Lista con variables segun target
# =================================

# Lista con variables para hipertensión
variables_hta = [
    "Sexo", "Edad", "IMC", "Nivel_Estudios", "Clase_Social", "Act_Fisica",
    "Hs_Sentado", "Tabaco_Consolidado", "Cons_Alcohol", "Ultimo_Chequeo_Tension",
    "Diabetes_Diag_Medico", "Cons_Fruta", "Cons_Carne", "Cons_Huevo",
    "Cons_Pescado", "Cons_Carbohidrato", "Cons_Cereales", "Cons_Verduras",
    "Cons_Legumbres", "Cons_Embutidos", "Cons_Lacteos", "Cons_Dulces",
    "Cons_Refrescos", "Cons_Comida_Rapida", "Cons_Snacks_Salados",
    "Cons_Zumos_Naturales"
]

# Lista con variables para diabetes
variables_diab = [
    "Sexo", "Edad", "IMC", "Nivel_Estudios", "Clase_Social", "Act_Fisica",
    "Hs_Sentado", "Tabaco_Consolidado", "Cons_Alcohol", "Ultimo_Chequeo_Tension",
    "Hipertension_Diag_Medico", "Cons_Fruta", "Cons_Carne", "Cons_Huevo",
    "Cons_Pescado", "Cons_Carbohidrato", "Cons_Cereales", "Cons_Verduras",
    "Cons_Legumbres", "Cons_Embutidos", "Cons_Lacteos", "Cons_Dulces",
    "Cons_Refrescos", "Cons_Comida_Rapida", "Cons_Snacks_Salados",
    "Cons_Zumos_Naturales"
]

# ========================================
# Asignar lista de variables a los target
# ========================================

X_train_hta = X_train_hta[variables_hta]
X_test_hta = X_test_hta[variables_hta]

X_train_diab = X_train_diab[variables_diab]
X_test_diab = X_test_diab[variables_diab]

# TRATAMIENTO DE VALORES NULOS Y CÓDIGOS 9 / 99 DE LAS VARIABLES PREDICTORAS

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Definir los grupos exactos 
vars_numericas = ["Hs_Sentado", "Edad"]

var_ordinal = {"IMC"}

vars_frecuencias = ["Cons_Fruta", "Cons_Carne", "Cons_Huevo",
    "Cons_Pescado", "Cons_Carbohidrato", "Cons_Cereales", "Cons_Verduras",
    "Cons_Legumbres", "Cons_Embutidos", "Cons_Lacteos", "Cons_Dulces",
    "Cons_Refrescos", "Cons_Comida_Rapida", "Cons_Snacks_Salados",
    "Cons_Zumos_Naturales", "Ultimo_Chequeo_Tension"]

vars_categoricas = ["Sexo", "Nivel_Estudios", "Clase_Social", "Act_Fisica", 
                    "Tabaco_Consolidado", "Cons_Alcohol"]